# Suite PLN — Plain

Conversion between values and plain data: the exact snapshot format, symbol assignment, single-object vs reachable
snapshots, deserialization through injected builders, and rejection of malformed snapshots with precise errors.

In [ ]:
import copy
import math

from mbse.Schemas.Framework import Plain, Proxies, Reachable, Schemas as S, Validators
from mbse.Schemas.Framework.Errors import DecodeError
from support import Fake, entries, raises, same_graph, text

Tag = S.OfObject.Builder().properties(text("label"), text("n", int), text("x", float), text("ok", bool),
                                      text("raw", bytes)).create()
Edge = S.OfRelation.Builder().links("src", "dst").properties(text("w", int), text("note")).create()
Tag = S.OfObject.Builder(Tag).relations(lambda r: r.name("out").of(Edge).me("src"),
                                        lambda r: r.name("in").of(Edge).me("dst")).update()
Proxies.register("Tag", Tag)
Proxies.register("Edge", Edge)
B = Proxies.Builders
FromPlain = Plain.FromPlain(B)

## PLN-01 · Natives: per-kind and dispatching entry points agree

In [ ]:
for schema, value, plain in [(S.OfNative.Data(int), 7, 7), (S.OfNative.Data(bytes), b"\x01", "AQ=="),
                             (S.OfNative.Data(float), math.inf, "Infinity"), (S.OfNative.Data(bool), False, False)]:
    assert Plain.ToPlain(schema, value) == Plain.ToPlain.OfNative(schema, value) == plain
    assert FromPlain(schema, plain) == FromPlain.OfNative(schema, plain) == value
with raises(NotImplementedError):
    Plain.ToPlain(S.OfUnion.Data(), 1)
with raises(NotImplementedError):
    FromPlain(S.OfIntersection.Data(), {})
with raises(NotImplementedError):
    Plain.ToPlain(Edge, None)  # relations are not serialized on their own

## PLN-02 · The exact snapshot format

In [ ]:
a = B.Tag().label("a").n(1).create()
b = B.Tag().label("b").out(lambda e: e.dst(a).w(5)).create()
assert Plain.ToPlain(Tag, b) == {
    "root": "s0",
    "objects": {"s0": {"label": "b", "out": [{"dst": {"$ref": "s1", "$schema": "Tag"}, "w": 5}]}},
}
assert Plain.ToPlain.Reachable(Tag, b) == {
    "root": "s0",
    "objects": {
        "s0": {"label": "b", "out": [{"dst": {"$ref": "s1", "$schema": "Tag"}, "w": 5}]},
        "s1": {"label": "a", "n": 1, "in": [{"src": {"$ref": "s0", "$schema": "Tag"}, "w": 5}]},
    },
}
assert Plain.ToPlain(Tag, B.Tag().create()) == {"root": "s0", "objects": {"s0": {}}}  # absent means omitted

## PLN-03 · Properties are written in schema order, entries in relation order

In [ ]:
t = B.Tag().raw(b"").ok(True).x(0.5).n(0).label("").create()
assert list(Plain.ToPlain(Tag, t)["objects"]["s0"]) == ["label", "n", "x", "ok", "raw"]
e = B.Tag().label("e").out(lambda x: x.note("n").w(1).dst(a)).create()
assert list(Plain.ToPlain(Tag, e)["objects"]["s0"]["out"][0]) == ["dst", "w", "note"]

## PLN-04 · Symbols follow first-reference order; the root is always s0

In [ ]:
hub = B.Tag().label("hub").out(lambda x: x.dst(B.Tag().label("c1").create())).out(
    lambda x: x.dst(B.Tag().label("c2").create())).create()
graph = Plain.ToPlain.Reachable(Tag, hub)
assert graph["root"] == "s0"
assert [graph["objects"][s]["label"] for s in ["s0", "s1", "s2"]] == ["hub", "c1", "c2"]
assert [o.values["label"] if isinstance(o, Fake) else o.label for o in Reachable.of(hub)] == ["hub", "c1", "c2"]

## PLN-05 · Single-object snapshots leave references unresolved; FromPlain rejects them

In [ ]:
single = Plain.ToPlain(Tag, hub)
assert list(single["objects"]) == ["s0"]
with raises(DecodeError, match="$.objects.s0.out[0].dst: unresolved reference 's1'"):
    FromPlain(Tag, single)
assert FromPlain(Tag, Plain.ToPlain(Tag, B.Tag().label("alone").create())).label == "alone"

## PLN-06 · Round trips build new objects; entries seen from both ends are not duplicated

In [ ]:
back = FromPlain.Reachable(Tag, graph)
assert back is not hub and back.label == "hub"
assert same_graph(Plain.ToPlain.Reachable(Tag, back), graph)
assert len(entries(Tag, back, "out")) == 2
assert Validators.Validate(B).Reachable(Tag, back) == []
lone = B.Tag().label("lone").n(3).create()
lone_back = FromPlain.OfObject(Tag, Plain.ToPlain.OfObject(Tag, lone))
assert lone_back is not lone and (lone_back.label, lone_back.n) == ("lone", 3)

## PLN-07 · Values at their edges survive a round trip

In [ ]:
edge = B.Tag().label("\x00 日本語 🚀 \ud800").n(-(2**100)).x(-0.0).ok(False).raw(bytes(range(256))).create()
edge_back = FromPlain(Tag, Plain.ToPlain(Tag, edge))
assert (edge_back.label, edge_back.n, edge_back.ok, edge_back.raw) == (edge.label, edge.n, False, edge.raw)
assert math.copysign(1, edge_back.x) == -1.0
for special in [math.nan, math.inf, -math.inf]:
    got = FromPlain(Tag, Plain.ToPlain(Tag, B.Tag().x(special).create())).x
    assert (math.isnan(got) and math.isnan(special)) or got == special

## PLN-08 · Finding F6 (pinned): a round trip can renumber symbols
Restoring re-adds each object's entries, which moves them to the end of the relation's table, so entry order and
therefore first-reference numbering can change. Content is preserved (compare up to renumbering).

In [ ]:
# The smallest case (found by search): n1 links to itself, then to n0; the snapshot is rooted at n0.
n0, n1 = B.Tag().label("n0").create(), B.Tag().label("n1").create()
B.Tag(n1).out(lambda x: x.dst(n1)).update()
B.Tag(n1).out(lambda x: x.dst(n0)).update()
before = Plain.ToPlain.Reachable(Tag, n0)
after = Plain.ToPlain.Reachable(Tag, FromPlain.Reachable(Tag, before))
assert same_graph(before, after)
assert before != after  # exact equality does not hold: n1's entries come back in a different order

## PLN-09 · ToPlain checks the root schema and native types

In [ ]:
with raises(TypeError, match="not an instance of the given schema"):
    Plain.ToPlain(S.OfObject.Data(), a)
with raises(TypeError, match="expected int, got str"):
    Plain.ToPlain(Tag, B.Tag().n("1").create())
with raises(TypeError):
    Plain.ToPlain.Reachable(Tag, B.Tag().out(lambda x: x.dst(B.Tag().n(True).create())).create())  # also when reached

## PLN-10 · Hand-written visitables serialize too

In [ ]:
fa = Fake("Tag", {"label": "fa"})
fb = Fake("Tag", {"label": "fb"}, {"out": [{"dst": fa, "w": 1}]})
fg = Plain.ToPlain.Reachable(Tag, fb)
assert fg["objects"]["s0"] == {"label": "fb", "out": [{"dst": {"$ref": "s1", "$schema": "Tag"}, "w": 1}]}
assert fg["objects"]["s1"] == {"label": "fa"}  # a fake only writes what it holds: no mirrored 'in' entry
restored = FromPlain.Reachable(Tag, fg)
assert [x["dst"].label for x in entries(Tag, restored, "out")] == ["fa"]
with raises(KeyError, match="unknown property"):
    Plain.ToPlain(Tag, Fake("Tag", {"colour": "red"}))
with raises(KeyError, match="unknown adjacency"):
    Plain.ToPlain(Tag, Fake("Tag", {}, {"sideways": [{}]}))

## PLN-11 · Malformed snapshots are rejected before anything is built, with a path and a reason

In [ ]:
ga = B.Tag().label("ga").create()
gb = B.Tag().label("gb").out(lambda x: x.dst(ga).w(5)).create()
good = Plain.ToPlain.Reachable(Tag, gb)  # exactly two objects, each referenced once


def mutated(change):
    snapshot = copy.deepcopy(good)
    change(snapshot)
    return snapshot


def link0(s):
    return s["objects"]["s0"]["out"][0]


SNAPSHOT = "$: expected an object snapshot: {'root': symbol, 'objects': {symbol: object}}"
REFERENCE = "$.objects.s0.out[0].dst: a reference is {'$ref': symbol, '$schema': name}"
cases = [
    ([], SNAPSHOT),
    ({"root": "s0"}, SNAPSHOT),
    ({"root": "s0", "objects": {}, "extra": 1}, SNAPSHOT),
    ({"root": "s0", "objects": []}, SNAPSHOT),
    ({"root": "zz", "objects": {"s0": {}}}, "$.root: root 'zz' is not in the snapshot's objects"),
    ({"root": ["s0"], "objects": {"s0": {}}}, "$.root: root ['s0'] is not in the snapshot's objects"),
    ({"root": "s0", "objects": {"s0": []}}, "$.objects.s0: an object must be a mapping, got list"),
    (mutated(lambda s: s["objects"]["s0"].__setitem__("out", "oops")), "$.objects.s0.out: an adjacency must be a list of entries"),
    (mutated(lambda s: s["objects"]["s0"]["out"].append("oops")), "$.objects.s0.out[1]: an entry must be a mapping, got str"),
    (mutated(lambda s: link0(s)["dst"].pop("$schema")), REFERENCE),
    (mutated(lambda s: link0(s)["dst"].__setitem__("extra", 1)), REFERENCE),
    (mutated(lambda s: link0(s)["dst"].__setitem__("$ref", 1)), REFERENCE),
    (mutated(lambda s: link0(s)["dst"].__setitem__("$schema", "Nope")), "$.objects.s1: no object schema registered as 'Nope'"),
    (mutated(lambda s: link0(s)["dst"].__setitem__("$schema", "Edge")), "$.objects.s1: no object schema registered as 'Edge'"),
    (mutated(lambda s: link0(s)["dst"].__setitem__("$ref", "s9")),
     "$.objects.s0.out[0].dst: unresolved reference 's9': the snapshot does not contain it"),
    (mutated(lambda s: link0(s).__setitem__("dst", "s1")), "$.objects.s0.out[0].dst: a link must be a reference"),
    (mutated(lambda s: link0(s).pop("dst")), "$.objects.s0.out[0]: links ['dst'] are not set"),
    (mutated(lambda s: link0(s).__setitem__("src", {"$ref": "s0", "$schema": "Tag"})),
     "$.objects.s0.out[0].src: 'src' is this object's own link, which is implied"),
    (mutated(lambda s: link0(s).__setitem__("colour", "red")),
     "$.objects.s0.out[0].colour: the relation has no link or property 'colour'"),
    (mutated(lambda s: link0(s).__setitem__("w", [1])), "$.objects.s0.out[0].w: expected int, got list"),
    (mutated(lambda s: s["objects"]["s0"].__setitem__("label", {"x": 1})), "$.objects.s0.label: expected str, got dict"),
    (mutated(lambda s: s["objects"]["s0"].__setitem__("colour", "red")),
     "$.objects.s0.colour: 'Tag' has no property or adjacency 'colour'"),
    (mutated(lambda s: s["objects"].__setitem__("s9", {})),
     "$.objects.s9: nothing references this object, so its schema is unknown"),
    (mutated(lambda s: s["objects"].__setitem__("my key", {})),
     '$.objects["my key"]: nothing references this object, so its schema is unknown'),
    (mutated(lambda s: s["objects"]["s1"]["in"][0]["src"].__setitem__("$schema", "Other")),
     "$.objects.s1.in[0].src: 's0' is referenced as both 'Tag' and 'Other'"),
]
for snapshot, message in cases:
    try:
        FromPlain.Reachable(Tag, snapshot)
    except DecodeError as error:
        assert str(error) == message, (str(error), message)
        assert error.path is not None and message == f"{error.path}: {error.reason}" and error.line is None
    else:
        raise AssertionError(message)

## PLN-11b · Ends that disagree are not malformed: each end's entries are restored
Each entry is written under every object it links. If the two ends of a snapshot disagree, both versions are
restored (the union of entries), since each is well-formed on its own. Validation is the place to catch this.

In [ ]:
asymmetric = mutated(lambda s: s["objects"]["s1"]["in"][0]["src"].__setitem__("$ref", "s1"))
restored_a = FromPlain.Reachable(Tag, asymmetric)
assert [x["dst"].label for x in entries(Tag, restored_a, "out")] == ["ga"]
ga_back = entries(Tag, restored_a, "out")[0]["dst"]
assert sorted(x["src"].label for x in entries(Tag, ga_back, "in")) == ["ga", "gb"]  # s0 -> s1 and s1 -> s1

## PLN-12 · Values of the wrong type are decoding errors with a path; a bad root schema is a program error

In [ ]:
with raises(DecodeError, match="$.objects.s0.out[0].w: expected int, got str"):
    FromPlain.Reachable(Tag, mutated(lambda s: link0(s).__setitem__("w", "5")))
with raises(DecodeError, match="$.objects.s0.label: expected str, got int"):
    FromPlain.Reachable(Tag, mutated(lambda s: s["objects"]["s0"].__setitem__("label", 5)))
with raises(DecodeError, match="$.objects.s1.raw: invalid base64 text"):
    FromPlain.Reachable(Tag, mutated(lambda s: s["objects"]["s1"].__setitem__("raw", "@@")))
with raises(DecodeError, match="$: expected int, got str"):
    FromPlain(S.OfNative.Data(int), "1")
with raises(TypeError, match="root schema must be an object schema"):
    FromPlain.Reachable(Edge, good)
with raises(TypeError, match="root schema must be an object schema"):
    FromPlain.Reachable(S.OfNative.Data(str), good)

## PLN-13 · A snapshot that is rejected builds nothing and leaves the input untouched

In [ ]:
bad = mutated(lambda s: link0(s).pop("dst"))
frozen = copy.deepcopy(bad)
before_ids = {o.identity() for o in Reachable.of(gb)}
with raises(ValueError):
    FromPlain.Reachable(Tag, bad)
assert bad == frozen
assert {o.identity() for o in Reachable.of(gb)} == before_ids

## PLN-14 · FromPlain only uses the builders it is given

In [ ]:
class Recording:
    def __init__(self, inner):
        self.inner, self.names = inner, []

    def schema(self, name):
        return self.inner.schema(name)

    def name_of(self, schema):
        return self.inner.name_of(schema)

    def __getattr__(self, name):
        self.names.append(name)
        return getattr(self.inner, name)


recording = Recording(B)
Plain.FromPlain(recording).Reachable(Tag, good)
assert recording.names and set(recording.names) == {"Tag"}

# A problem anywhere, even a bad value in the last object, is found before any builder is asked for.
for bad in [mutated(lambda s: s["objects"]["s1"].__setitem__("label", 5)), mutated(lambda s: link0(s).__setitem__("w", "5"))]:
    untouched = Recording(B)
    with raises(DecodeError):
        Plain.FromPlain(untouched).Reachable(Tag, bad)
    assert untouched.names == []


class Failing(Recording):
    def schema(self, name):
        raise RuntimeError("registry down")


with raises(RuntimeError, match="registry down"):  # only lookup failures become "no object schema registered"
    Plain.FromPlain(Failing(B)).Reachable(Tag, good)

## PLN-15 · Property schemas that are not native are refused; per-kind text entry points

In [ ]:
Holder = S.OfObject.Builder().properties(lambda p: p.name("inner").of(Tag), text("label")).create()
Proxies.register("Holder", Holder)
holder = B.Holder().inner("anything").create()  # the builder does not check (F8)
with raises(TypeError, match="property 'inner' is not native"):
    Plain.ToPlain(Holder, holder)
with raises(NotImplementedError, match="property 'inner': only native values are supported by Plain yet"):
    FromPlain(Holder, {"root": "s0", "objects": {"s0": {"inner": "x"}}})
with raises(NotImplementedError, match="str is not supported by Plain yet"):
    Plain.ToPlain("not a schema", 1)
with raises(NotImplementedError, match="str is not supported by Plain yet"):
    FromPlain("not a schema", 1)

from mbse.Schemas.Framework import JSON, YAML

lone = B.Tag().label("lone-text").n(9).create()
assert JSON.FromJSON(B).OfObject(Tag, JSON.ToJSON.OfObject(Tag, lone)).n == 9
assert YAML.FromYAML(B).OfObject(Tag, YAML.ToYAML.OfObject(Tag, lone)).n == 9